# ClimX : émulateur par analogues forcés, pleine résolution

Ce notebook télécharge les données 192×288, en extrait les cellules terrestres, émule SSP2-4.5 et écrit
des fichiers de soumission de plus en plus précis (12, puis 24, puis 48 réalisations).

**Avant de lancer**
1. Le notebook doit être rattaché à la compétition `climx-main-track`.
2. Panneau *Settings* : **Internet ON**, *Accelerator* : **None**.
3. Cliquez sur **Save Version → Save & Run All** : l'exécution continue une fois le navigateur fermé.

**Après l'exécution**, ouvrez l'onglet *Output* de la version enregistrée et soumettez le fichier
`submission_native_k48_m*.parquet` le plus grand (`m` est le nombre de réalisations).

Si l'exécution s'arrête en route, les fichiers déjà écrits restent valables.


In [ ]:
# Réglages. Les valeurs par défaut conviennent à une première exécution.
STAGES = [12, 24, 48]      # réalisations : un fichier de soumission à chaque étape
N_ANALOGUES = 48           # années candidates par année cible
RUNS = ["hist", "ssp370", "ssp126", "ssp585"]  # ordre de priorité si le disque manque
RUN_VALIDATION = True      # à la fin, score local avec SSP3-7.0 exclu, si le temps le permet
VALIDATION_REALIZATIONS = 8
TIME_BUDGET_HOURS = 11.0   # marge sous la limite de session de Kaggle


In [ ]:
import os, shutil, signal, subprocess, sys, time, glob, json, urllib.request
from pathlib import Path

START = time.time()
WORK = Path("/kaggle/temp/climx"); WORK.mkdir(parents=True, exist_ok=True)   # hors de la sortie (limitée en taille)
OUTPUT = Path("/kaggle/working/outputs"); OUTPUT.mkdir(parents=True, exist_ok=True)
CODE = Path("/kaggle/temp/code"); CODE.mkdir(parents=True, exist_ok=True)

free = shutil.disk_usage(WORK).free / 1e9
ram = int(open("/proc/meminfo").read().split()[1]) / 1e6
WORKERS = os.cpu_count() or 4
print(f"Python {sys.version.split()[0]} | {WORKERS} CPU | RAM {ram:.0f} GB | disque libre {free:.0f} GB")
# Pic : magasin complet (35 Go) + un fichier source (33 Go). Avec moins, les derniers scénarios seront ignorés.
if free < 70:
    print(f"ATTENTION : {free:.0f} GB libres pour 70 GB souhaités ; les derniers scénarios seront ignorés.")
assert free > 55, "Disque insuffisant pour l'historique et un scénario."
if ram < 12:
    WORKERS = 2
try:
    urllib.request.urlopen("https://huggingface.co", timeout=30)
except Exception as error:
    raise RuntimeError("Internet inaccessible : activez Internet ON dans Settings, puis relancez.") from error
print("Internet OK")


## Code du projet
Les modules sont écrits tels quels depuis le projet local.


In [ ]:
(CODE / 'common.py').write_text('"""Data access and official-code wrappers shared by validation and submission scripts."""\nfrom __future__ import annotations\n\nimport glob\nimport os\nimport sys\nimport warnings\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom pathlib import Path\n\nos.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")\n\nimport cftime\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[1]\nREPO = Path(os.environ.get("CLIMX_REPO", ROOT / "repo_ClimX"))\nDATA = Path(os.environ.get("CLIMX_LITE", ROOT / "data" / "lite"))\nOUT = Path(os.environ.get("CLIMX_OUT", ROOT / "outputs"))\nsys.path.insert(0, str(REPO))\n\nimport rnNSE  # noqa: E402  (official scorer)\nfrom src import indices_xclim as I  # noqa: E402  (official index code)\nfrom src.utils import kaggle_submission as ks  # noqa: E402\nfrom src.utils.indices_utils import format_indices  # noqa: E402\nfrom src.utils.original_data_values import LAT, LON  # noqa: E402\n\nTARGETS = ["huss", "pr", "psl", "sfcWind", "tas", "tasmax", "tasmin"]\nTARGET_UNITS = {"huss": "1", "pr": "kg m-2 s-1", "psl": "Pa", "sfcWind": "m s-1", "tas": "K", "tasmax": "K", "tasmin": "K"}\nGHG = ["CO2_LBC", "CH4_LBC", "N2O_LBC", "CFC11eq_LBC", "CF2CL2_LBC"]\nAEROSOLS = ["BC_AX", "BC_N", "SO2", "SO4_PR", "OM_NI"]\nTRAIN_SCENARIOS = ["ssp126", "ssp370", "ssp585"]\nDAYS = 365\n\n\n@dataclass\nclass Run:\n    """One continuous simulation: daily targets by year plus annual forcing series."""\n\n    name: str\n    years: np.ndarray\n    fields: dict[str, np.ndarray] | None  # var -> (n_years, 365, lat, lon)\n    ghg: dict[str, np.ndarray]  # var -> (n_years,)\n    aerosols: dict[str, np.ndarray]  # var -> (n_years,) global area-weighted mean\n\n\ndef _files(pattern: str) -> list[str]:\n    found = sorted(glob.glob(str(DATA / pattern)))\n    if not found:\n        raise FileNotFoundError(f"No file matching {pattern} under {DATA}")\n    return found\n\n\ndef _squeeze_scenario(ds: xr.Dataset) -> xr.Dataset:\n    if "forcing_scenario" in ds.dims:\n        ds = ds.isel(forcing_scenario=0, drop=True)\n    return ds.drop_vars("forcing_scenario", errors="ignore")\n\n\ndef open_hist() -> xr.Dataset:\n    """Historical run, loaded in memory, with CF units kept for the official index code."""\n    parts = [_squeeze_scenario(xr.open_dataset(f)) for f in _files("*hist_train_*.nc")]\n    daily = xr.concat([p[TARGETS] for p in parts], dim="time").load()\n    # Each file already holds the full 1850-2014 annual and monthly forcing series.\n    forcings = parts[0][GHG + AEROSOLS].load()\n    for p in parts:\n        p.close()\n    return xr.merge([daily, forcings])\n\n\ndef open_ssp(scenario: str) -> xr.Dataset:\n    pattern = f"*test_forcings_{scenario}.nc" if scenario == "ssp245" else f"*train_{scenario}.nc"\n    with xr.open_dataset(_files(pattern)[0]) as ds:\n        return _squeeze_scenario(ds).load()\n\n\ndef _area_weights(lat: np.ndarray) -> np.ndarray:\n    w = np.cos(np.deg2rad(lat))\n    return w / w.sum()\n\n\ndef _to_year_blocks(da: xr.DataArray, years: np.ndarray) -> np.ndarray:\n    """Reshape a noleap daily array to (year, 365, lat, lon). A truncated final day is repeated."""\n    values = da.transpose("time", "lat", "lon").values.astype(np.float32)\n    stamp = da["time"].dt.year.values\n    out = np.empty((len(years), DAYS) + values.shape[1:], dtype=np.float32)\n    for i, year in enumerate(years):\n        block = values[stamp == year]\n        if len(block) < DAYS:\n            pad = np.repeat(block[-1:], DAYS - len(block), axis=0)\n            block = np.concatenate([block, pad])\n        out[i] = block\n    return out\n\n\ndef to_run(name: str, ds: xr.Dataset) -> Run:\n    years = np.unique(ds["time_year"].dt.year.values)\n    w = _area_weights(ds["lat"].values)\n    ghg = {v: ds[v].values.astype(np.float64) for v in GHG}\n    aerosols = {}\n    for v in AEROSOLS:\n        # Global, annual mean emission flux: the forcing\'s spatial pattern is not used yet.\n        spatial = (ds[v].values * w[None, :, None]).sum(axis=1).mean(axis=1)\n        aerosols[v] = pd.Series(spatial).groupby(ds["time_month"].dt.year.values).mean().values\n    fields = None\n    if "tas" in ds:\n        fields = {v: _to_year_blocks(ds[v], years) for v in TARGETS}\n    return Run(name, years, fields, ghg, aerosols)\n\n\n@lru_cache(maxsize=4)\ndef time_axis(first_year: int, last_year: int) -> xr.CFTimeIndex:\n    """Noleap daily axis at 12:00, ending on Dec 30 of the last year like the dataset."""\n    stamps = [\n        cftime.DatetimeNoLeap(y, 1, 1, 12) + pd.Timedelta(days=d)\n        for y in range(first_year, last_year + 1)\n        for d in range(DAYS)\n    ]\n    return xr.CFTimeIndex(stamps[:-1])\n\n\ndef blocks_to_dataset(fields: dict[str, np.ndarray], first_year: int, lat, lon, units: dict) -> xr.Dataset:\n    """Wrap (year, 365, lat, lon) arrays as a daily dataset with the dataset\'s calendar and units."""\n    n_years = next(iter(fields.values())).shape[0]\n    time = time_axis(first_year, first_year + n_years - 1)\n    data = {}\n    for var, arr in fields.items():\n        flat = arr.reshape((-1,) + arr.shape[2:])[: len(time)]\n        data[var] = xr.DataArray(flat, dims=("time", "lat", "lon"), attrs={"units": units[var]})\n    return xr.Dataset(data, coords={"time": time, "lat": lat, "lon": lon})\n\n\ndef compute_indices(daily: xr.Dataset, hist: xr.Dataset) -> xr.Dataset:\n    """Compute the 15 indices with the official functions, then the official annualisation."""\n    needed = daily[[v for v in ("tas", "tasmax", "tasmin", "pr") if v in daily]]\n    out = {}\n    with warnings.catch_warnings():\n        warnings.simplefilter("ignore")\n        for name, func in I.SELECTED_INDICES.items():\n            kwargs = {"historical_data_path": hist} if "historical_data_path" in func.__code__.co_varnames else {}\n            out[name] = func(needed, **kwargs).compute()\n        out = format_indices(out, "emulator", "lite", save_indices=False)\n    return xr.Dataset({k: v.drop_vars("quantile", errors="ignore") for k, v in out.items()})\n\n\ndef upscale_nearest(indices: xr.Dataset) -> xr.Dataset:\n    """Nearest-neighbour upscaling to the competition grid, as in the starter notebook."""\n    lat = np.asarray(LAT, dtype=float)\n    lon = np.asarray(LON, dtype=float)\n    return indices.sortby("lat").sortby("lon").interp(\n        lat=lat, lon=lon, method="nearest", kwargs={"fill_value": "extrapolate"}\n    )\n\n\ndef kaggle_frame(indices: xr.Dataset) -> pd.DataFrame:\n    """Build the row_id + 15 columns frame through the official conversion."""\n    return ks.indices_xr_to_kaggle_dataframe(indices, index_columns=ks.CLIMX_INDEX_COLUMNS)\n\n\ndef official_scores(truth: xr.Dataset, pred: xr.Dataset) -> dict[str, float]:\n    """Official scorer overall and per index, on two lite-grid index sets upscaled the same way.\n\n    Truth is the lite-grid truth, so this ranks models but does not estimate the Kaggle score,\n    which compares against native-resolution truth.\n    """\n    sol = kaggle_frame(upscale_nearest(truth))\n    sub = kaggle_frame(upscale_nearest(pred))\n    scores = {"overall": rnNSE.score(sol.copy(), sub.copy(), "row_id")}\n    for col in ks.CLIMX_INDEX_COLUMNS:\n        scores[col] = rnNSE.score(sol[["row_id", col]].copy(), sub[["row_id", col]].copy(), "row_id")\n    return scores\n', encoding='utf8')
print('common.py', 'écrit')


In [ ]:
(CODE / 'model.py').write_text('"""Forced-analogue emulator: daily fields from forcings only.\n\n1. A small energy-balance regression turns GHG and aerosol forcings into a forced global\n   temperature for any scenario year.\n2. Per cell, month and variable, a regression gives the forced monthly mean as a function of\n   that forced state.\n3. A realisation of a target year copies a whole library year with a similar forced state,\n   then shifts it by the difference between the target and library forced means. Daily weather,\n   spells, tails, and consistency across cells and variables come from the library year.\n\nAveraging indices over realisations estimates their expected value given the forcing, which\nis what minimises the squared error in the metric.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom common import DAYS, TARGETS, Run\n\nMULTIPLICATIVE = {"pr", "huss", "sfcWind"}\nPR_FLOOR = 1e-9\nMONTH_LENGTHS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])\nMONTH_OF_DAY = np.repeat(np.arange(12), MONTH_LENGTHS)\nMONTH_CENTERS = np.cumsum(MONTH_LENGTHS) - MONTH_LENGTHS / 2\n\n\ndef _monthly_to_daily(monthly: np.ndarray) -> np.ndarray:\n    """Periodic linear interpolation of a (12, ...) array to (365, ...)."""\n    days = np.arange(DAYS) + 0.5\n    x = np.concatenate([[MONTH_CENTERS[-1] - DAYS], MONTH_CENTERS, [MONTH_CENTERS[0] + DAYS]])\n    pos = np.clip(np.searchsorted(x, days) - 1, 0, len(x) - 2)\n    frac = ((days - x[pos]) / (x[pos + 1] - x[pos])).reshape((-1,) + (1,) * (monthly.ndim - 1))\n    extended = np.concatenate([monthly[-1:], monthly, monthly[:1]])\n    return extended[pos] * (1 - frac) + extended[pos + 1] * frac\n\n\ndef _ghg_forcing(run: Run, reference: Run) -> np.ndarray:\n    """Simplified radiative forcing (W m-2), relative to the reference\'s start.\n\n    The dataset stores mole fractions; the CH4 and N2O coefficients expect ppb.\n    """\n    co2, ch4, n2o = (run.ghg[v] for v in ("CO2_LBC", "CH4_LBC", "N2O_LBC"))\n    co2_0, ch4_0, n2o_0 = (reference.ghg[v][0] for v in ("CO2_LBC", "CH4_LBC", "N2O_LBC"))\n    return (\n        5.35 * np.log(co2 / co2_0)\n        + 0.036 * (np.sqrt(ch4 * 1e9) - np.sqrt(ch4_0 * 1e9))\n        + 0.12 * (np.sqrt(n2o * 1e9) - np.sqrt(n2o_0 * 1e9))\n    )\n\n\ndef _lagged(series: np.ndarray, tau: float, start: float = 0.0) -> np.ndarray:\n    """First-order low-pass filter: the response of a heat reservoir with time scale tau."""\n    out = np.empty_like(series)\n    state = start\n    k = 1.0 / tau\n    for i, value in enumerate(series):\n        state = state + k * (value - state)\n        out[i] = state\n    return out\n\n\n@dataclass\nclass Config:\n    n_analogues: int = 48  # library years a target year can draw from\n    taus: tuple[float, ...] = (4.0, 60.0)  # response time scales of the energy-balance fit (years)\n    use_aerosols: bool = True\n    aerosol_weight: float = 0.5  # weight of aerosol distance in analogue selection\n    quadratic: bool = True  # allow forced means to curve with warming\n    apply_shift: bool = True  # False: copy library years without adjusting them\n    seed: int = 0\n\n\n@dataclass\nclass ForcedAnalogueEmulator:\n    config: Config = field(default_factory=Config)\n\n    def _state(self, hist: Run, run: Run | None = None) -> np.ndarray:\n        """Forcing predictors of a run, continued from the end of the historical run if a scenario."""\n        f_hist = _ghg_forcing(hist, hist)\n        a_hist = np.log(self._aerosol(hist))\n        if run is None:\n            f, a = f_hist, a_hist\n            n = 0\n        else:\n            f = np.concatenate([f_hist, _ghg_forcing(run, hist)])\n            a = np.concatenate([a_hist, np.log(self._aerosol(run))])\n            n = len(f_hist)\n        cols = [_lagged(f, tau)[n:] for tau in self.config.taus]\n        cols.append(_lagged(a, 2.0, start=a[0])[n:])\n        return np.column_stack(cols)\n\n    def _aerosol(self, run: Run) -> np.ndarray:\n        # SO2 dominates the aerosol cooling; BC_AX and OM_NI are kept out of this first version.\n        return np.maximum(run.aerosols["SO2"], 1e-30)\n\n    def fit(self, hist: Run, library: list[Run], lat: np.ndarray) -> "ForcedAnalogueEmulator":\n        self.hist = hist\n        self.library = [hist] + library\n        cos = np.cos(np.deg2rad(lat))\n        self.weights = cos / cos.sum()\n\n        # Forced global temperature from all runs (energy-balance regression).\n        states = [self._state(hist)] + [self._state(hist, r) for r in library]\n        self.lib_runs = np.concatenate([np.full(len(r.years), i) for i, r in enumerate(self.library)])\n        self.lib_years = np.concatenate([np.arange(len(r.years)) for r in self.library])\n        s = np.concatenate(states)\n        if not self.config.use_aerosols:\n            s = s[:, :-1]\n        self.state_mean, self.state_std = s.mean(0), s.std(0)\n        gmst = np.concatenate([self._gmst(r) for r in self.library])\n        design = self._design_state(s)\n        self.ebm, *_ = np.linalg.lstsq(design, gmst, rcond=None)\n        self.lib_forced = design @ self.ebm\n        self.lib_aero = np.concatenate(states)[:, -1]\n        self.aero_scale = self.lib_aero.std() or 1.0\n        self.ebm_rmse = float(np.sqrt(np.mean((gmst - self.lib_forced) ** 2)))\n\n        # Forced monthly means, per cell and variable, as a function of the forced state.\n        design_means = self._design_means(self.lib_forced, self.lib_aero)\n        self.coef = {}\n        for var in TARGETS:\n            monthly = np.concatenate([self._monthly(r.fields[var], var) for r in self.library])\n            n, months, nlat, nlon = monthly.shape\n            flat = monthly.reshape(n, -1)\n            coef, *_ = np.linalg.lstsq(design_means, flat, rcond=None)\n            self.coef[var] = coef.reshape((design_means.shape[1], months, nlat, nlon)).astype(np.float64)\n        return self\n\n    def _gmst(self, run: Run) -> np.ndarray:\n        annual = run.fields["tas"].mean(axis=1)\n        return (annual.mean(axis=2) * self.weights).sum(axis=1)\n\n    def _design_state(self, s: np.ndarray) -> np.ndarray:\n        return np.column_stack([np.ones(len(s)), (s - self.state_mean) / self.state_std])\n\n    def _design_means(self, forced: np.ndarray, aero: np.ndarray) -> np.ndarray:\n        t = forced - self.lib_forced_ref()\n        cols = [np.ones_like(t), t]\n        if self.config.quadratic:\n            cols.append(t**2)\n        if self.config.use_aerosols:\n            cols.append((aero - self.lib_aero.mean()) / self.aero_scale)\n        return np.column_stack(cols)\n\n    def lib_forced_ref(self) -> float:\n        return float(self.lib_forced.mean())\n\n    def _monthly(self, block: np.ndarray, var: str) -> np.ndarray:\n        """(year, 365, lat, lon) -> (year, 12, lat, lon) monthly means, log for positive variables."""\n        out = np.stack([block[:, MONTH_OF_DAY == m].mean(axis=1) for m in range(12)], axis=1)\n        return np.log(np.maximum(out, PR_FLOOR)) if var in MULTIPLICATIVE else out\n\n    def forced_state(self, run: Run) -> tuple[np.ndarray, np.ndarray]:\n        s = self._state(self.hist, run)\n        aero = s[:, -1]\n        if not self.config.use_aerosols:\n            s = s[:, :-1]\n        return self._design_state(s) @ self.ebm, aero\n\n    def _forced_daily(self, forced: float, aero: float, var: str) -> np.ndarray:\n        """Forced mean of one year as a (365, lat, lon) array, in the variable\'s regression space."""\n        d = self._design_means(np.array([forced]), np.array([aero]))[0]\n        return _monthly_to_daily(np.tensordot(d, self.coef[var], axes=(0, 0)))\n\n    def choose_analogues(self, forced: np.ndarray, aero: np.ndarray, available: np.ndarray | None = None) -> list[np.ndarray]:\n        """For each target year, the library entries closest in forced state, among those available."""\n        chosen = []\n        for f, a in zip(forced, aero):\n            dist = np.abs(self.lib_forced - f)\n            if self.config.use_aerosols:\n                dist = dist + self.config.aerosol_weight * np.abs(self.lib_aero - a) / self.aero_scale\n            if available is not None:\n                dist = np.where(available, dist, np.inf)\n            chosen.append(np.argsort(dist, kind="stable")[: self.config.n_analogues])\n        return chosen\n\n    def realization(self, run: Run, r: int, variables: list[str] = TARGETS) -> dict[str, np.ndarray]:\n        """One daily realisation of a scenario as var -> (year, 365, lat, lon)."""\n        forced, aero = self.forced_state(run)\n        chosen = self.choose_analogues(forced, aero)\n        out = {v: None for v in variables}\n        for y, candidates in enumerate(chosen):\n            # A per-year permutation so that realisations use distinct library years.\n            order = np.random.default_rng([self.config.seed, y]).permutation(len(candidates))\n            entry = candidates[order[r % len(candidates)]]\n            source = self.library[self.lib_runs[entry]]\n            year = self.lib_years[entry]\n            for var in variables:\n                daily = source.fields[var][year].astype(np.float64)\n                if self.config.apply_shift:\n                    delta = self._forced_daily(forced[y], aero[y], var) - self._forced_daily(\n                        self.lib_forced[entry], self.lib_aero[entry], var\n                    )\n                    daily = daily * np.exp(delta) if var in MULTIPLICATIVE else daily + delta\n                if out[var] is None:\n                    out[var] = np.empty((len(forced),) + daily.shape, dtype=np.float32)\n                out[var][y] = daily\n        # Shifts are per variable: restore tasmin <= tas <= tasmax.\n        if {"tas", "tasmax", "tasmin"} <= set(variables):\n            out["tasmax"] = np.maximum(out["tasmax"], out["tas"])\n            out["tasmin"] = np.minimum(out["tasmin"], out["tas"])\n        return out\n', encoding='utf8')
print('model.py', 'écrit')


In [ ]:
(CODE / 'native.py').write_text('"""Tile store and emulation of land cells at full resolution.\n\nThe native files hold one gzip chunk per global day, so everything scored is extracted once into\nper-tile arrays: land cells only, and the four variables the official indices read. The forced\nstate and the analogue choice come from the lite data and are identical for all tiles, so tiles\nof a realisation stay coherent with each other.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom pathlib import Path\n\nimport cftime\nimport numpy as np\nimport xarray as xr\n\nfrom common import DAYS, LAT, LON, TARGET_UNITS, compute_indices, rnNSE, time_axis\nfrom model import MONTH_CENTERS, MONTH_OF_DAY, MULTIPLICATIVE, PR_FLOOR\n\nINDEX_VARS = ("tas", "tasmax", "tasmin", "pr")\nTEMP_OFFSET, TEMP_SCALE = 150.0, 0.005  # K: uint16 covers 150-477 K at 0.005 K, half the disk of float32\nTILE_CELLS = 1500\nBASELINE = (1961, 1990)\nMAX_LOG_SHIFT = np.log(2.0)  # guards arid cells, whose monthly means are mostly noise\n\n\n@dataclass(frozen=True)\nclass Grid:\n    name: str\n    lat: np.ndarray\n    lon: np.ndarray\n    cells: np.ndarray  # flat indices (lat, lon order) of the cells the emulator handles\n\n    @property\n    def shape(self) -> tuple[int, int]:\n        return len(self.lat), len(self.lon)\n\n    def tiles(self, tile_cells: int = TILE_CELLS) -> list[np.ndarray]:\n        return np.array_split(np.arange(len(self.cells)), int(np.ceil(len(self.cells) / tile_cells)))\n\n\ndef native_grid() -> Grid:\n    """Competition grid, restricted to the land cells of the official scorer\'s region mask."""\n    mask, _ = rnNSE._decode_mask()\n    return Grid("native", np.asarray(LAT, float), np.asarray(LON, float), np.flatnonzero(mask.ravel() >= 0))\n\n\ndef lite_grid(lat: np.ndarray, lon: np.ndarray) -> Grid:\n    """All lite cells: runs the same code on the small dataset, to test the pipeline."""\n    return Grid("lite", np.asarray(lat, float), np.asarray(lon, float), np.arange(len(lat) * len(lon)))\n\n\ndef _encode(var: str, values: np.ndarray) -> np.ndarray:\n    if var == "pr":\n        return values.astype(np.float32)\n    return np.rint((values - TEMP_OFFSET) / TEMP_SCALE).astype(np.uint16)\n\n\ndef _decode(var: str, stored: np.ndarray) -> np.ndarray:\n    if var == "pr":\n        return np.asarray(stored, dtype=np.float32)\n    return TEMP_OFFSET + TEMP_SCALE * np.asarray(stored, dtype=np.float32)\n\n\nclass TileStore:\n    """(year, day, cell) arrays on disk, one file per run, variable and tile."""\n\n    def __init__(self, root: Path, grid: Grid, tile_cells: int = TILE_CELLS):\n        self.root = Path(root)\n        self.grid = grid\n        self.tile_ids = grid.tiles(tile_cells)\n\n    def _path(self, run: str, var: str, tile: int) -> Path:\n        return self.root / run / f"{var}_{tile:03d}.bin"\n\n    def _meta_path(self, run: str) -> Path:\n        return self.root / run / "meta.json"\n\n    def years(self, run: str) -> np.ndarray | None:\n        path = self._meta_path(run)\n        if not path.exists():\n            return None\n        meta = json.loads(path.read_text())\n        return np.arange(meta["first_year"], meta["first_year"] + meta["done_years"])\n\n    def _open(self, run: str, var: str, tile: int, n_years: int, mode: str) -> np.memmap:\n        shape = (n_years, DAYS, len(self.tile_ids[tile]))\n        dtype = np.float32 if var == "pr" else np.uint16\n        return np.memmap(self._path(run, var, tile), dtype=dtype, mode=mode, shape=shape)\n\n    def extract(self, run: str, files: list[Path], first_year: int, last_year: int) -> None:\n        """Copy land cells of the given NetCDF files, year by year. Resumes from the last complete year."""\n        n_years = last_year - first_year + 1\n        (self.root / run).mkdir(parents=True, exist_ok=True)\n        meta_path = self._meta_path(run)\n        meta = json.loads(meta_path.read_text()) if meta_path.exists() else None\n        if meta is None:\n            meta = {"first_year": first_year, "n_years": n_years, "done_years": 0}\n            for var in INDEX_VARS:\n                for tile in range(len(self.tile_ids)):\n                    self._open(run, var, tile, n_years, "w+").flush()\n            meta_path.write_text(json.dumps(meta))\n        for file in files:\n            with xr.open_dataset(file, chunks={}) as ds:\n                ds = ds.isel(forcing_scenario=0, drop=True) if "forcing_scenario" in ds.dims else ds\n                assert np.allclose(ds["lat"].values, self.grid.lat, atol=1e-3), "Unexpected latitude order"\n                assert np.allclose(ds["lon"].values, self.grid.lon, atol=1e-3), "Unexpected longitude order"\n                stamps = ds["time"].dt.year.values\n                for year in np.unique(stamps):\n                    y = int(year) - first_year\n                    if y < meta["done_years"] or y >= n_years:\n                        continue\n                    picked = np.flatnonzero(stamps == year)\n                    for var in INDEX_VARS:\n                        block = ds[var].isel(time=slice(picked[0], picked[-1] + 1)).transpose("time", "lat", "lon").values\n                        flat = block.reshape(block.shape[0], -1)[:, self.grid.cells]\n                        if len(flat) < DAYS:  # the final year stops on Dec 30\n                            flat = np.concatenate([flat, np.repeat(flat[-1:], DAYS - len(flat), axis=0)])\n                        coded = _encode(var, flat)\n                        for tile, ids in enumerate(self.tile_ids):\n                            out = self._open(run, var, tile, n_years, "r+")\n                            out[y] = coded[:, ids]\n                            out.flush()\n                    meta["done_years"] = y + 1\n                    meta_path.write_text(json.dumps(meta))\n                    print(f"{run} {year} extracted", flush=True)\n\n    def read_year(self, run: str, tile: int, year_index: int, variables=INDEX_VARS) -> dict[str, np.ndarray]:\n        n_years = json.loads(self._meta_path(run).read_text())["n_years"]\n        return {v: _decode(v, self._open(run, v, tile, n_years, "r")[year_index]) for v in variables}\n\n    def monthly(self, run: str, tile: int) -> dict[str, np.ndarray]:\n        """Monthly means per year in regression space, cached next to the tile files."""\n        cache = self.root / run / f"monthly_{tile:03d}.npz"\n        if cache.exists():\n            with np.load(cache) as data:\n                return {v: data[v] for v in INDEX_VARS}\n        n_years = len(self.years(run))\n        out = {}\n        for var in INDEX_VARS:\n            months = np.empty((n_years, 12, len(self.tile_ids[tile])), dtype=np.float32)\n            for y in range(n_years):\n                daily = self.read_year(run, tile, y, (var,))[var]\n                months[y] = np.stack([daily[MONTH_OF_DAY == m].mean(axis=0) for m in range(12)])\n            out[var] = np.log(np.maximum(months, PR_FLOOR)) if var in MULTIPLICATIVE else months\n        np.savez(cache, **out)\n        return out\n\n\n@lru_cache(maxsize=1)\ndef daily_weights() -> np.ndarray:\n    """(365, 12) periodic linear interpolation weights from monthly to daily values."""\n    centers = np.concatenate([[MONTH_CENTERS[-1] - DAYS], MONTH_CENTERS, [MONTH_CENTERS[0] + DAYS]])\n    weights = np.zeros((DAYS, 12))\n    for d in range(DAYS):\n        pos = np.clip(np.searchsorted(centers, d + 0.5) - 1, 0, len(centers) - 2)\n        frac = (d + 0.5 - centers[pos]) / (centers[pos + 1] - centers[pos])\n        weights[d, (pos - 1) % 12] += 1 - frac\n        weights[d, pos % 12] += frac\n    return weights\n\n\ndef tile_dataset(fields: dict[str, np.ndarray], first_year: int, n_years: int) -> xr.Dataset:\n    """Wrap (year, 365, cell) arrays with the dataset\'s calendar, including the missing last day."""\n    time = time_axis(first_year, first_year + n_years - 1)\n    data = {\n        v: xr.DataArray(a.reshape((-1, a.shape[2]))[: len(time)], dims=("time", "cell"), attrs={"units": TARGET_UNITS[v]})\n        for v, a in fields.items()\n    }\n    return xr.Dataset(data, coords={"time": time})\n\n\ndef baseline_dataset(store: TileStore, tile: int) -> xr.Dataset:\n    """1961-1990 of the historical run: the base period of the official percentile thresholds."""\n    years = store.years("hist")\n    picked = np.arange(BASELINE[0], BASELINE[1] + 1) - years[0]\n    assert picked[0] >= 0 and picked[-1] < len(years), "The baseline years have not been extracted"\n    blocks = [store.read_year("hist", tile, int(y), ("tasmax", "tasmin", "pr")) for y in picked]\n    fields = {v: np.stack([b[v] for b in blocks]) for v in blocks[0]}\n    n = fields["pr"].shape[0] * DAYS\n    time = xr.cftime_range(cftime.DatetimeNoLeap(BASELINE[0], 1, 1, 12), periods=n, freq="D", calendar="noleap")\n    return xr.Dataset(\n        {v: xr.DataArray(a.reshape((n, -1)), dims=("time", "cell"), attrs={"units": TARGET_UNITS[v]}) for v, a in fields.items()},\n        coords={"time": time},\n    )\n\n\nclass TileEmulator:\n    """Forced-analogue emulation of one tile, from a plan saved by run_native.make_plan."""\n\n    def __init__(self, store: TileStore, plan: dict, tile: int):\n        self.store = store\n        self.plan = plan\n        self.tile = tile\n        monthly = {}\n        runs = list(dict.fromkeys(plan["entry_run"].tolist()))\n        by_run = {run: store.monthly(run, tile) for run in runs}\n        first = {run: int(store.years(run)[0]) for run in runs}\n        for var in INDEX_VARS:\n            monthly[var] = np.stack(\n                [by_run[r][var][int(y) - first[r]] for r, y in zip(plan["entry_run"], plan["entry_year"])]\n            )\n        pinv = np.linalg.pinv(plan["design_entries"])\n        self.coef = {v: np.tensordot(pinv, monthly[v], axes=(1, 0)) for v in INDEX_VARS}\n        self.first = first\n\n    def realization(self, r: int) -> dict[str, np.ndarray]:\n        plan = self.plan\n        weights = daily_weights()\n        n_years = len(plan["target_years"])\n        out = {\n            v: np.empty((n_years, DAYS, len(self.store.tile_ids[self.tile])), dtype=np.float32) for v in INDEX_VARS\n        }\n        for y in range(n_years):\n            candidates = plan["candidates"][y]\n            order = np.random.default_rng([int(plan["seed"]), y]).permutation(len(candidates))\n            entry = int(candidates[order[r % len(candidates)]])\n            run = str(plan["entry_run"][entry])\n            year = int(plan["entry_year"][entry]) - self.first[run]\n            source = self.store.read_year(run, self.tile, year)\n            gap = plan["design_target"][y] - plan["design_entries"][entry]\n            for var in INDEX_VARS:\n                daily = source[var]\n                if plan["apply_shift"]:\n                    delta = weights @ np.tensordot(gap, self.coef[var], axes=(0, 0))\n                    if var in MULTIPLICATIVE:\n                        daily = daily * np.exp(np.clip(delta, -MAX_LOG_SHIFT, MAX_LOG_SHIFT))\n                    else:\n                        daily = daily + delta\n                out[var][y] = daily\n        out["tasmax"] = np.maximum(out["tasmax"], out["tas"])\n        out["tasmin"] = np.minimum(out["tasmin"], out["tas"])\n        return out\n\n    def indices(self, r: int, baseline: xr.Dataset) -> xr.Dataset:\n        ds = tile_dataset(self.realization(r), int(self.plan["target_years"][0]), len(self.plan["target_years"]))\n        return compute_indices(ds, baseline)\n\n\ndef truth_indices(store: TileStore, run: str, tile: int, baseline: xr.Dataset) -> xr.Dataset:\n    """Official indices of a held-out scenario, for validation."""\n    years = store.years(run)\n    blocks = [store.read_year(run, tile, y) for y in range(len(years))]\n    fields = {v: np.stack([b[v] for b in blocks]) for v in INDEX_VARS}\n    return compute_indices(tile_dataset(fields, int(years[0]), len(years)), baseline)\n', encoding='utf8')
print('native.py', 'écrit')


In [ ]:
(CODE / 'make_submission.py').write_text('"""Train on all scenarios, emulate SSP2-4.5 and write the Kaggle submission file.\n\nExample: python climx_emulator/make_submission.py --realizations 48\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport time\n\nimport numpy as np\nimport pandas as pd\n\nfrom common import OUT, TARGET_UNITS, TARGETS, TRAIN_SCENARIOS, blocks_to_dataset, compute_indices, ks, open_hist, open_ssp, rnNSE, to_run, upscale_nearest\nfrom model import Config, ForcedAnalogueEmulator\n\nN_ROWS = 86 * 192 * 288\n\n\ndef check_submission(path) -> None:\n    """Reread the written file and check it against the documented format and the official scorer."""\n    df = pd.read_parquet(path)\n    expected = ["row_id", *ks.CLIMX_INDEX_COLUMNS]\n    assert list(df.columns) == expected, f"Unexpected columns: {list(df.columns)}"\n    assert len(df) == N_ROWS, f"Expected {N_ROWS} rows, found {len(df)}"\n    assert (df["row_id"].values == np.arange(N_ROWS)).all(), "row_id is not 0..N-1 in order"\n    values = df[list(ks.CLIMX_INDEX_COLUMNS)].to_numpy(dtype=np.float64)\n    assert np.isfinite(values).all(), "Submission contains NaN or infinite values"\n    ks.validate_kaggle_submission_dataframe(df)\n    self_score = rnNSE.score(df.copy(), df.copy(), "row_id")\n    assert abs(self_score - 1.0) < 1e-12, f"Official scorer self-check returned {self_score}"\n    print(f"Format checks passed; official scorer accepts the file (self-score {self_score:.6f}).")\n    print(df[list(ks.CLIMX_INDEX_COLUMNS)].describe().loc[["min", "mean", "max"]].T.round(2).to_string())\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--realizations", type=int, default=48)\n    parser.add_argument("--n-analogues", type=int, default=Config.n_analogues)\n    parser.add_argument("--no-aerosols", action="store_true")\n    parser.add_argument("--no-quadratic", action="store_true")\n    parser.add_argument("--name", default="submission_lite_forced_analogue")\n    args = parser.parse_args()\n\n    start = time.time()\n    hist_ds = open_hist()\n    lat, lon = hist_ds["lat"].values, hist_ds["lon"].values\n    hist = to_run("hist", hist_ds)\n    library = [to_run(s, open_ssp(s)) for s in TRAIN_SCENARIOS]\n    test_ds = open_ssp("ssp245")\n    test_time = test_ds["time"].values\n    target = to_run("ssp245", test_ds)\n\n    config = Config(\n        n_analogues=args.n_analogues,\n        use_aerosols=not args.no_aerosols,\n        quadratic=not args.no_quadratic,\n    )\n    emulator = ForcedAnalogueEmulator(config).fit(hist, library, lat)\n    forced, _ = emulator.forced_state(target)\n    print(f"Forced GMST SSP2-4.5: {forced[0]:.2f} K (2015) -> {forced[-1]:.2f} K (2100)", flush=True)\n\n    OUT.mkdir(exist_ok=True)\n    total = None\n    for r in range(args.realizations):\n        # The emulator always produces all seven daily target variables; indices read four of them.\n        daily = emulator.realization(target, r, variables=TARGETS)\n        ds = blocks_to_dataset(daily, int(target.years[0]), lat, lon, TARGET_UNITS)\n        if r == 0:\n            assert (ds["time"].values == test_time).all(), "Emulated time axis differs from the test file"\n            encoding = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}\n            ds.to_netcdf(OUT / "daily_fields_ssp245_realisation0.nc", encoding=encoding)\n        indices = compute_indices(ds, hist_ds)\n        total = indices if total is None else total + indices\n        print(f"realisation {r + 1}/{args.realizations} ({time.time() - start:.0f}s)", flush=True)\n    mean_indices = total / args.realizations\n    # The official writer silently turns NaN into 0, so reject them before it runs.\n    missing = {k: int(v.isnull().sum()) for k, v in mean_indices.items() if v.isnull().any()}\n    assert not missing, f"NaN in computed indices: {missing}"\n    mean_indices.to_netcdf(OUT / "indices_ssp245_lite.nc")\n\n    path = OUT / f"{args.name}.parquet"\n    ks.write_kaggle_submission_parquet(upscale_nearest(mean_indices), path, cast_integer_columns=False)\n    check_submission(path)\n    (OUT / f"{args.name}.json").write_text(json.dumps(vars(args), indent=2))\n    print(f"Wrote {path} ({path.stat().st_size / 1e6:.0f} MB) in {time.time() - start:.0f}s")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf8')
print('make_submission.py', 'écrit')


In [ ]:
(CODE / 'run_native.py').write_text('"""Full-resolution pipeline: extract, emulate tile by tile, score or write the submission.\n\nEvery step can resume: extraction by year, emulation by tile and realisation.\n\n    python run_native.py fetch    # download, verify, extract, delete each source file\n    python run_native.py validate --holdout ssp370 --realizations 12\n    python run_native.py submit --realizations 48\n"""\nfrom __future__ import annotations\n\nimport os\n\nfor _name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n    os.environ.setdefault(_name, "1")\n\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport time\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport numpy as np\nimport xarray as xr\n\nfrom common import (\n    DATA, I, OUT, TRAIN_SCENARIOS, TARGET_UNITS, blocks_to_dataset, compute_indices, ks, official_scores, open_hist,\n    open_ssp, rnNSE, to_run, upscale_nearest,\n)\nfrom model import Config, ForcedAnalogueEmulator\nfrom native import INDEX_VARS, Grid, TileEmulator, TileStore, baseline_dataset, lite_grid, native_grid, truth_indices\n\nWORK = Path(os.environ.get("CLIMX_WORK", OUT.parent / "work"))\nREPO_ID = "isp-uv-es/ClimX"\nFULL_FILES = {\n    # Earlier history is far colder than SSP2-4.5, so it would never be chosen as an analogue.\n    "hist": ("NorESM2-MM_r1i1p1f1_with_forcing_hist_train_1932_2014.nc", 1932, 2014),\n    "ssp370": ("NorESM2-MM_r1i1p1f1_with_forcing_train_ssp370.nc", 2015, 2100),\n    "ssp126": ("NorESM2-MM_r1i1p1f1_with_forcing_train_ssp126.nc", 2015, 2100),\n    "ssp585": ("NorESM2-MM_r1i1p1f1_with_forcing_train_ssp585.nc", 2015, 2100),\n}\nINDEX_COLUMNS = list(ks.CLIMX_INDEX_COLUMNS)\n\n\ndef get_grid(name: str) -> Grid:\n    if name == "native":\n        return native_grid()\n    with xr.open_dataset(next(DATA.glob("*test_forcings_ssp245.nc"))) as ds:\n        return lite_grid(ds["lat"].values, ds["lon"].values)\n\n\ndef store_for(grid: Grid) -> TileStore:\n    return TileStore(WORK / f"store_{grid.name}", grid)\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1 << 22), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef fetch(args) -> None:\n    """Download each file, check it, extract land cells and delete it, keeping peak disk use low."""\n    from huggingface_hub import HfApi, hf_hub_download\n\n    grid = get_grid(args.grid)\n    store = store_for(grid)\n    if args.grid == "lite":\n        hist_files = sorted(DATA.glob("*hist_train_*.nc"))\n        store.extract("hist", hist_files, 1850, 2014)\n        for scenario in TRAIN_SCENARIOS:\n            store.extract(scenario, list(DATA.glob(f"*train_{scenario}.nc")), 2015, 2100)\n        return\n\n    info = HfApi().dataset_info(REPO_ID, files_metadata=True)\n    sizes = {s.rfilename: s for s in info.siblings}\n    downloads = WORK / "downloads"\n    for run in args.runs:\n        name, first, last = FULL_FILES[run]\n        years = store.years(run)\n        if years is not None and len(years) == last - first + 1:\n            print(f"{run}: already extracted", flush=True)\n            continue\n        remote = sizes[f"full/{name}"]\n        free = shutil.disk_usage(WORK).free\n        need = remote.size + 16e9\n        if free < need:\n            assert run != "hist", f"{free / 1e9:.0f} GB free: the historical run is required for the baselines"\n            print(f"{run}: SKIPPED, {free / 1e9:.0f} GB free and {need / 1e9:.0f} GB needed", flush=True)\n            continue\n        started = time.time()\n        for attempt in range(50):\n            try:\n                path = Path(hf_hub_download(REPO_ID, f"full/{name}", repo_type="dataset", local_dir=downloads))\n                break\n            except Exception as error:  # flaky connections: the next attempt resumes the partial file\n                print(f"{run}: attempt {attempt + 1} failed ({error}); retrying", flush=True)\n                time.sleep(30)\n        else:\n            raise RuntimeError(f"Could not download {name}")\n        print(f"{run}: downloaded {remote.size / 1e9:.1f} GB in {time.time() - started:.0f}s", flush=True)\n        assert _sha256(path) == remote.lfs.sha256, f"Checksum mismatch for {name}"\n        store.extract(run, [path], first, last)\n        path.unlink()\n\n\ndef make_plan(args, grid: Grid, store: TileStore, holdout: str | None) -> Path:\n    """Forced state and analogue choices from the lite data, common to all tiles."""\n    hist_ds = open_hist()\n    hist = to_run("hist", hist_ds)\n    train = {s: to_run(s, open_ssp(s)) for s in TRAIN_SCENARIOS}\n    target = train.pop(holdout) if holdout else to_run("ssp245", open_ssp("ssp245"))\n    config = Config(n_analogues=args.n_analogues, apply_shift=not args.no_shift)\n    model = ForcedAnalogueEmulator(config).fit(hist, list(train.values()), hist_ds["lat"].values)\n\n    names = np.array([model.library[i].name for i in model.lib_runs])\n    calendar = np.array([model.library[i].years[y] for i, y in zip(model.lib_runs, model.lib_years)])\n    available = np.zeros(len(names), dtype=bool)\n    for run in set(names.tolist()):\n        years = store.years(run)\n        if years is None:\n            print(f"Run {run} has not been extracted: not used as a library", flush=True)\n            continue\n        available |= (names == run) & np.isin(calendar, years)\n    assert available.sum() >= config.n_analogues, f"Only {available.sum()} library years available"\n    forced, aero = model.forced_state(target)\n    chosen = model.choose_analogues(forced, aero, available)\n    position = np.cumsum(available) - 1  # library entry -> position among extracted entries\n    plan = {\n        "entry_run": names[available],\n        "entry_year": calendar[available],\n        "design_entries": model._design_means(model.lib_forced, model.lib_aero)[available],\n        "design_target": model._design_means(forced, aero),\n        "candidates": np.array([position[c] for c in chosen]),\n        "target_years": target.years,\n        "seed": config.seed,\n        "apply_shift": config.apply_shift,\n    }\n    path = _run_dir(args, grid, holdout) / "plan.npz"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez(path, **plan)\n    print(f"Plan: {available.sum()} library years; forced GMST {forced[0]:.2f} -> {forced[-1]:.2f} K", flush=True)\n    return path\n\n\ndef _run_dir(args, grid: Grid, holdout: str | None) -> Path:\n    label = f"{holdout or \'ssp245\'}_k{args.n_analogues}{args.tag}"\n    return WORK / f"runs_{grid.name}" / label\n\n\ndef _as_array(indices: xr.Dataset) -> np.ndarray:\n    return np.stack([indices[c].transpose("time", "cell").values for c in INDEX_COLUMNS]).astype(np.float64)\n\n\ndef run_tile(grid_name: str, plan_path: str, tile: int, realizations: int, holdout: str | None) -> tuple[int, float]:\n    """Add realisations to the running sum of a tile, saving after each one."""\n    grid = get_grid(grid_name)\n    store = store_for(grid)\n    with np.load(plan_path) as data:\n        plan = {k: data[k] for k in data.files}\n    folder = Path(plan_path).parent\n    state_path = folder / f"sum_{tile:03d}.npz"\n    total, done = None, 0\n    if state_path.exists():\n        with np.load(state_path) as state:\n            total, done = state["total"], int(state["count"])\n    started = time.time()\n    if done >= realizations and (holdout is None or (folder / f"truth_{tile:03d}.npy").exists()):\n        return tile, 0.0\n    # The official percentile cache is keyed by id(dataset): a later tile could reuse the id.\n    I._PERCENTILE_CACHE.clear()\n    baseline = baseline_dataset(store, tile)\n    if holdout and not (folder / f"truth_{tile:03d}.npy").exists():\n        np.save(folder / f"truth_{tile:03d}.npy", _as_array(truth_indices(store, holdout, tile, baseline)))\n    emulator = TileEmulator(store, plan, tile)\n    for r in range(done, realizations):\n        values = _as_array(emulator.indices(r, baseline))\n        assert np.isfinite(values).all(), f"Non-finite index in tile {tile}, realisation {r}"\n        total = values if total is None else total + values\n        # One file renamed into place, so an interruption cannot separate the sum from its count.\n        temp = folder / f"sum_{tile:03d}.tmp.npz"\n        np.savez(temp, total=total, count=r + 1)\n        os.replace(temp, state_path)\n    return tile, time.time() - started\n\n\ndef _run_all(args, grid: Grid, plan_path: Path, holdout: str | None) -> None:\n    store = store_for(grid)\n    tiles = range(len(store.tile_ids)) if args.max_tiles is None else range(args.max_tiles)\n    started = time.time()\n    with ProcessPoolExecutor(args.workers) as pool:\n        jobs = [pool.submit(run_tile, grid.name, str(plan_path), t, args.realizations, holdout) for t in tiles]\n        for n, job in enumerate(as_completed(jobs), 1):\n            tile, seconds = job.result()\n            print(f"tile {tile}: {seconds:.0f}s ({n}/{len(jobs)} tiles, {time.time() - started:.0f}s total)", flush=True)\n\n\ndef assemble(folder: Path, grid: Grid, store: TileStore, prefix: str, years: np.ndarray, fill: xr.Dataset | None, n: int) -> xr.Dataset:\n    """Tile files to a dataset of index maps on the grid; cells outside the tiles come from fill."""\n    nlat, nlon = grid.shape\n    maps = {c: np.zeros((len(years), nlat * nlon)) for c in INDEX_COLUMNS}\n    if fill is not None:\n        for c in INDEX_COLUMNS:\n            # float64: land results are copied into this array and some indices are stored as float32.\n            maps[c] = fill[c].transpose("time", "lat", "lon").values.reshape(len(years), -1).astype(np.float64)\n    for tile, ids in enumerate(store.tile_ids):\n        if prefix == "sum":\n            path = folder / f"sum_{tile:03d}.npz"\n            if not path.exists():\n                continue\n            with np.load(path) as state:\n                assert int(state["count"]) == n, f"Tile {tile} has {int(state[\'count\'])} realisations, expected {n}"\n                values = state["total"] / n\n        else:\n            path = folder / f"{prefix}_{tile:03d}.npy"\n            if not path.exists():\n                continue\n            values = np.load(path)\n        for i, c in enumerate(INDEX_COLUMNS):\n            maps[c][:, grid.cells[ids]] = values[i]\n    time_values = np.array([np.datetime64(f"{int(y)}-01-01", "ns") for y in years])\n    return xr.Dataset(\n        {c: (("time", "lat", "lon"), m.reshape(len(years), nlat, nlon)) for c, m in maps.items()},\n        coords={"time": time_values, "lat": grid.lat, "lon": grid.lon},\n    )\n\n\ndef validate(args) -> None:\n    grid = get_grid(args.grid)\n    store = store_for(grid)\n    plan_path = _run_dir(args, grid, args.holdout) / "plan.npz"\n    if not plan_path.exists():\n        make_plan(args, grid, store, args.holdout)\n    _run_all(args, grid, plan_path, args.holdout)\n    years = np.load(plan_path)["target_years"]\n    pred = assemble(plan_path.parent, grid, store, "sum", years, None, args.realizations)\n    truth = assemble(plan_path.parent, grid, store, "truth", years, None, 1)\n    if args.max_tiles is not None:\n        print(f"Partial run: {args.max_tiles} tiles. Unprocessed cells are zero in both maps and score as invalid.")\n    if grid.name == "lite":\n        scores = official_scores(truth, pred)\n    else:\n        sol, sub = ks.indices_xr_to_kaggle_dataframe(truth, index_columns=ks.CLIMX_INDEX_COLUMNS), ks.indices_xr_to_kaggle_dataframe(pred, index_columns=ks.CLIMX_INDEX_COLUMNS)\n        scores = {"overall": rnNSE.score(sol.copy(), sub.copy(), "row_id")}\n        for c in INDEX_COLUMNS:\n            scores[c] = rnNSE.score(sol[["row_id", c]].copy(), sub[["row_id", c]].copy(), "row_id")\n    print(f"Validation {args.holdout}, grid {grid.name}: overall {scores[\'overall\']:+.4f}")\n    print("  " + "  ".join(f"{k} {v:+.2f}" for k, v in scores.items() if k != "overall"))\n    OUT.mkdir(exist_ok=True)\n    path = OUT / f"native_validation_{grid.name}_{args.holdout}_k{args.n_analogues}_m{args.realizations}{args.tag}.json"\n    path.write_text(json.dumps({"config": {k: v for k, v in vars(args).items() if k != "func"}, "scores": scores}, indent=2))\n\n\ndef ocean_fill(realizations: int = 8) -> xr.Dataset:\n    """Index maps for unscored cells, from the seven-variable lite emulator and the official code."""\n    hist_ds = open_hist()\n    lat, lon = hist_ds["lat"].values, hist_ds["lon"].values\n    hist = to_run("hist", hist_ds)\n    library = [to_run(s, open_ssp(s)) for s in TRAIN_SCENARIOS]\n    target = to_run("ssp245", open_ssp("ssp245"))\n    model = ForcedAnalogueEmulator(Config()).fit(hist, library, lat)\n    total = None\n    for r in range(realizations):\n        daily = model.realization(target, r, variables=list(INDEX_VARS))\n        indices = compute_indices(blocks_to_dataset(daily, int(target.years[0]), lat, lon, TARGET_UNITS), hist_ds)\n        total = indices if total is None else total + indices\n    return upscale_nearest(total / realizations)\n\n\ndef submit(args) -> None:\n    from make_submission import check_submission\n\n    grid = get_grid(args.grid)\n    assert grid.name == "native", "Only the native grid can be submitted through this script"\n    store = store_for(grid)\n    plan_path = _run_dir(args, grid, None) / "plan.npz"\n    if not plan_path.exists():\n        make_plan(args, grid, store, None)\n    _run_all(args, grid, plan_path, None)\n    folder = plan_path.parent\n    missing_tiles = [t for t in range(len(store.tile_ids)) if not (folder / f"sum_{t:03d}.npz").exists()]\n    assert not missing_tiles, f"Tiles without results: {missing_tiles}"\n    years = np.load(plan_path)["target_years"]\n    fill_path = folder / "ocean_fill.nc"\n    if not fill_path.exists():\n        ocean_fill().transpose("time", "lat", "lon").to_netcdf(fill_path)\n    with xr.open_dataset(fill_path) as fill:\n        maps = assemble(folder, grid, store, "sum", years, fill.load(), args.realizations)\n    missing = {c: int(maps[c].isnull().sum()) for c in INDEX_COLUMNS if maps[c].isnull().any()}\n    assert not missing, f"NaN in computed indices: {missing}"\n    OUT.mkdir(exist_ok=True)\n    path = OUT / f"submission_native_k{args.n_analogues}_m{args.realizations}{args.tag}.parquet"\n    ks.write_kaggle_submission_parquet(maps, path, cast_integer_columns=False)\n    check_submission(path)\n    print(f"Wrote {path} ({path.stat().st_size / 1e6:.0f} MB)", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    sub = parser.add_subparsers(dest="command", required=True)\n    fetch_p = sub.add_parser("fetch")\n    fetch_p.add_argument("--runs", nargs="+", default=list(FULL_FILES))\n    fetch_p.set_defaults(func=fetch)\n    for name, func in (("validate", validate), ("submit", submit)):\n        p = sub.add_parser(name)\n        p.add_argument("--realizations", type=int, default=48)\n        p.add_argument("--n-analogues", type=int, default=48)\n        p.add_argument("--workers", type=int, default=4)\n        p.add_argument("--no-shift", action="store_true")\n        p.add_argument("--max-tiles", type=int, default=None, help="process only the first tiles (benchmark)")\n        p.add_argument("--tag", default="")\n        p.set_defaults(func=func)\n        if name == "validate":\n            p.add_argument("--holdout", choices=TRAIN_SCENARIOS, default="ssp370")\n    for p in (fetch_p, *sub.choices.values()):\n        if not any(a.dest == "grid" for a in p._actions):\n            p.add_argument("--grid", choices=["native", "lite"], default="native")\n    args = parser.parse_args()\n    WORK.mkdir(parents=True, exist_ok=True)\n    args.func(args)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf8')
print('run_native.py', 'écrit')


## Environnement


In [ ]:
# Environnement isolé : Python 3.11 et les versions du dépôt officiel (l'index officiel dépend de xclim 0.59.1).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
VENV = Path("/kaggle/temp/venv")
subprocess.run(UV + ["venv", str(VENV), "--python", "3.11", "--clear"], check=True)
PY = str(VENV / "bin" / "python")
(CODE / "requirements.txt").write_text('xarray==2024.7.0\ndask==2024.8.1\nnumpy==2.1.1\npandas==2.2.3\nscipy==1.13.1\nnetcdf4==1.7.1\nh5netcdf==1.3.0\ncftime==1.6.4\nbottleneck\npyarrow==17.0.0\ncf-xarray==0.10.8\nxclim==0.59.1\npint-xarray\nhuggingface_hub\ntqdm\n')
subprocess.run(UV + ["pip", "install", "-q", "--python", PY, "-r", str(CODE / "requirements.txt")], check=True)

# Code officiel (indices, métrique, format), à un commit fixé.
REPO = Path("/kaggle/temp/repo_ClimX")
if not (REPO / "rnNSE.py").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    for command in (["init", "-q"],
                    ["fetch", "-q", "--depth", "1", "https://github.com/IPL-UV/ClimX.git", "c1b2eab8c9f79bda04c6455dc4745335dc4dc5ab"],
                    ["checkout", "-q", "FETCH_HEAD"]):
        subprocess.run(["git", "-C", str(REPO)] + command, check=True)

# Données lite (état forcé et choix des analogues) : fournies par la compétition, sinon Hugging Face.
ENV = dict(os.environ, CLIMX_REPO=str(REPO), CLIMX_WORK=str(WORK), CLIMX_OUT=str(OUTPUT), HF_HUB_DISABLE_XET="1")
found = glob.glob("/kaggle/input/**/*subsampled_16_test_forcings_ssp245.nc", recursive=True)
if found:
    ENV["CLIMX_LITE"] = str(Path(found[0]).parent)
else:
    print("Données lite non montées : téléchargement depuis Hugging Face.")
    script = "from huggingface_hub import snapshot_download as s; s('isp-uv-es/ClimX', repo_type='dataset', local_dir='/kaggle/temp/lite_data', allow_patterns=['lite/*'])"
    subprocess.run([PY, "-c", script], check=True, env=ENV)
    ENV["CLIMX_LITE"] = "/kaggle/temp/lite_data/lite"
print("Données lite :", ENV["CLIMX_LITE"])
print(subprocess.run([PY, "-c", "import xclim, xarray, numpy; print('xclim', xclim.__version__, 'xarray', xarray.__version__, 'numpy', numpy.__version__)"], capture_output=True, text=True, check=True).stdout)


In [ ]:
SCRIPT = str(CODE / "run_native.py")
RUN_CWD = Path("/kaggle/temp/cwd"); RUN_CWD.mkdir(exist_ok=True)
LOG = OUTPUT / "run_log.txt"

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def run(*args):
    """Lance une étape avec journal en direct. Renvoie False si elle échoue ou dépasse le temps restant."""
    limit = remaining()
    if limit <= 60:
        print("Temps écoulé : étape ignorée :", args)
        return False
    command = [PY, "-u", SCRIPT, *map(str, args)]
    print(f"\n=== {' '.join(map(str, args))} ({limit / 3600:.1f} h restantes) ===", flush=True)
    started = time.time()
    process = subprocess.Popen(command, cwd=RUN_CWD, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    tail = []
    with open(LOG, "a") as log:
        try:
            for line in process.stdout:
                log.write(line); log.flush()
                tail = (tail + [line])[-30:]
                if "Warning" not in line and "warn" not in line:
                    print(line, end="", flush=True)
                if time.time() - started > limit:
                    raise TimeoutError
            code = process.wait()
        except TimeoutError:
            os.killpg(process.pid, signal.SIGKILL)
            print("Temps écoulé : étape arrêtée.")
            return False
    print(f"=== terminé en {(time.time() - started) / 60:.0f} min, code {code} ===", flush=True)
    if code != 0:
        print("".join(tail))
    return code == 0


## Données pleine résolution
Téléchargement, contrôle d'empreinte, extraction, suppression de la source, fichier par fichier.


In [ ]:
assert run("fetch", "--runs", *RUNS), "Échec du téléchargement ou de l'extraction : voir le journal ci-dessus."
store = WORK / "store_native"
for name in RUNS:
    meta = store / name / "meta.json"
    print(name, json.loads(meta.read_text()) if meta.exists() else "non extrait")
print(f"Disque libre : {shutil.disk_usage(WORK).free / 1e9:.0f} GB ; temps écoulé : {(time.time() - START) / 3600:.1f} h")


## Émulation et fichiers de soumission


In [ ]:
written = []
for realizations in STAGES:
    if not run("submit", "--realizations", realizations, "--n-analogues", N_ANALOGUES, "--workers", WORKERS):
        break
    written.append(realizations)
    print("Fichiers prêts :", sorted(p.name for p in OUTPUT.glob("*.parquet")))
assert written, "Aucun fichier de soumission écrit : voir le journal ci-dessus."


## Validation


In [ ]:
# Score local en pleine résolution, avec SSP3-7.0 exclu : il ne prévoit pas le score Kaggle, mais détaille les indices.
if RUN_VALIDATION:
    run("validate", "--holdout", "ssp370", "--realizations", VALIDATION_REALIZATIONS, "--n-analogues", N_ANALOGUES, "--workers", WORKERS)


In [ ]:
for path in sorted(OUTPUT.iterdir()):
    print(f"{path.name:50s} {path.stat().st_size / 1e6:9.1f} MB")
for path in sorted(OUTPUT.glob("native_validation_*.json")):
    scores = json.loads(path.read_text())["scores"]
    print("\n" + path.name, f"\n  global {scores['overall']:+.4f}")
    print("  " + "  ".join(f"{k} {v:+.2f}" for k, v in scores.items() if k != "overall"))
print(f"\nDurée totale : {(time.time() - START) / 3600:.1f} h")
print("À soumettre : onglet Output, le fichier submission_native avec le plus grand m.")
